# CellRank

https://github.com/theislab/cellrank/issues/592

In [ ]:
import cellrank as cr

import scanpy as sc
import scvelo as scv

import pickle

from scipy import sparse
import pandas as pd
import numpy as np
import math

import matplotlib.pyplot as plt

import os

## Settings

In [ ]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20250213HSCT')

In [ ]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning) 

# Warnings scanpy off
sc.settings.verbosity = 0

## Plotting 

In [ ]:
# Scanpy figures
cm_to_inch = lambda x: x / 2.54
sc.set_figure_params(figsize=(cm_to_inch(10), cm_to_inch(10)), dpi_save=300, fontsize=16, frameon=False, color_map='Reds', facecolor='white')
scv.set_figure_params(figsize=(cm_to_inch(10), cm_to_inch(10)), dpi_save=300, fontsize=16, frameon=False, color_map='Reds', facecolor='white')

# Helper functions

In [ ]:
# PCA 
def pca_workflow(adata, n_comps): 
    
    adata = adata.raw.to_adata()
    
    sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=2000, subset=False)
    
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    sc.pp.pca(adata, n_comps=n_comps, use_highly_variable=True)
    
    X_pca = adata.obsm['X_pca']
    
    return(X_pca)

In [ ]:
# Velocity kernel
def vk_workflow(adata, suffix, compute=False):
    
    if compute: 
        
        # Set genes to include for moment computation 
        adata.var['m_genes'] = adata.var.index.isin(scv.pp.filter_and_normalize(adata, min_shared_counts=5, subset_highly_variable=False, n_top_genes=None, log=False, copy=True).var_names)

        print(sum(adata.var['m_genes']))
        
        # Compute PCA 
        adata.obsm['X_pca'] = pca_workflow(adata.copy(), n_comps=30)
        
        # Compute neighbors on PCA 
        sc.pp.neighbors(adata, n_neighbors=30, n_pcs=30, use_rep='X_pca')
        
        # Normalize X, spliced, and unspliced
        sc.pp.normalize_total(adata, target_sum=1e6)
        sc.pp.log1p(adata)

        # Subset by var names filter
        adata = adata[:, adata.var['m_genes']].copy()

        # First-/second-order moments are computed for each cell across its nearest neighbors
        scv.pp.moments(adata, n_pcs=None, n_neighbors=None)
        scv.tl.recover_dynamics(adata)

        # Estimates velocities in a gene-specific manner
        scv.tl.velocity(adata, mode='dynamical')

        # CellRank kernel 
        from cellrank.kernels import VelocityKernel
        vk = VelocityKernel(adata)
        
        vk.write('data/object/cellrank/kernel/vk'+suffix+'.pickle', write_adata=True)
        
    else: 
        
        from cellrank.kernels import VelocityKernel
        vk = VelocityKernel.read(fname='data/object/cellrank/kernel/vk'+suffix+'.pickle')
        
    return vk

In [ ]:
# Connectivity kernel
def ck_workflow(adata, suffix, use_rep='latent', compute=False):
    
    if compute: 
        
        # Neighbor graph on latent space 
        sc.pp.neighbors(adata, n_neighbors=30, n_pcs=None, use_rep=use_rep)
        
        # CellRank kernel 
        from cellrank.kernels import ConnectivityKernel
        ck = ConnectivityKernel(adata)
        
        ck.write('data/object/cellrank/kernel/ck'+suffix+'.pickle', write_adata=True)
        
    else: 
        
        from cellrank.kernels import ConnectivityKernel
        ck = ConnectivityKernel.read(fname='data/object/cellrank/kernel/ck'+suffix+'.pickle')
        
    return ck

# CD8+ T-cells

In [ ]:
# Load integrated data
adata = sc.read_h5ad('data/object/scvi/pseudotime/adata_1.h5ad')

In [ ]:
# Only use Skin cells
adata = adata[adata.obs.hto_demux_class=='Skin'].copy()

## Velocity kernel

In [ ]:
adata_v = sc.read_h5ad('data/object/velocyto.h5ad') 

In [ ]:
obs_names = adata.obs_names
var_names = adata.var_names.intersection(adata_v.var_names)

In [ ]:
adata = adata[obs_names, var_names]
adata_v = adata_v[obs_names, var_names]

In [ ]:
adata.X = adata_v.X
adata.layers = adata_v.layers

In [ ]:
vk = vk_workflow(adata=adata.copy(), suffix='_0', compute=True)

## AUC connectivity kernel

In [ ]:
auc = pd.read_csv("result/scenic/res_0/auc_mtx.csv", index_col=0)
auc = auc.loc[adata.obs_names]
adata_auc = sc.AnnData(X=auc.values, obs=adata.obs.copy(), var=pd.DataFrame(index=auc.columns))

In [ ]:
ck = ck_workflow(adata=adata_auc.copy(), use_rep='X', suffix='_0', compute=True)